# Cleaned, Full Hazard Model

In [49]:
# load libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

import pyhere as here
from sklearn.preprocessing import StandardScaler

# set display options for better readability
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [72]:
import random
import numpy as np
import torch
import torch.nn as nn

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### Load Data

In [51]:
### Load data 

# load clinical metadata
clinical = pd.read_csv(here.here("data", "processed","clinical_data.csv"))
# load expression data
expression_data = pd.read_csv(here.here("data", "processed","expression_data.csv"))
# mutation binary data
mutation_binary = pd.read_csv(here.here("data", "processed","mutation_binary_data.csv"))
# mutation classified data
mutation_classified = pd.read_csv(here.here("data", "processed","mutation_classified_data.csv"))

In [52]:
# define outcome variables
df = clinical.copy()
time_col = "overall_survival_months"
event_col = "death_from_cancer"

# 'event' will be 1 if died of cancer of disease, 0 otherwise (including died of other causes or alive)
df["event"] = (df[event_col] == "Died of Disease").astype(int)
df["time"] = df[time_col]

In [53]:
# view nan values
missing_counts = df.isna().sum()
print(missing_counts)

patient_id                          0
age_at_diagnosis                    0
type_of_breast_surgery             22
cancer_type                         0
cancer_type_detailed               15
cellularity                        54
chemotherapy                        0
pam50_+_claudin-low_subtype         0
cohort                              0
er_status_measured_by_ihc          30
er_status                           0
neoplasm_histologic_grade          72
her2_status_measured_by_snp6        0
her2_status                         0
tumor_other_histologic_subtype     15
hormone_therapy                     0
inferred_menopausal_state           0
integrative_cluster                 0
primary_tumor_laterality          106
lymph_nodes_examined_positive       0
mutation_count                     45
nottingham_prognostic_index         0
oncotree_code                      15
overall_survival_months             0
overall_survival                    0
pr_status                           0
radio_therap

In [54]:
df["event"].value_counts()

event
0    1282
1     622
Name: count, dtype: int64

In [55]:
df.head()

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,er_status,neoplasm_histologic_grade,her2_status_measured_by_snp6,her2_status,tumor_other_histologic_subtype,hormone_therapy,inferred_menopausal_state,integrative_cluster,primary_tumor_laterality,lymph_nodes_examined_positive,mutation_count,nottingham_prognostic_index,oncotree_code,overall_survival_months,overall_survival,pr_status,radio_therapy,3-gene_classifier_subtype,tumor_size,tumor_stage,death_from_cancer,event,time
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,1,Negative,1,ER-/HER2-,22.0,2.0,Living,0,140.500000
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,1,Positive,1,ER+/HER2- High Prolif,10.0,1.0,Living,0,84.633333
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,1,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,0,Positive,0,unknown,15.0,2.0,Died of Disease,1,163.700000
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,1,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,1,Positive,1,unknown,25.0,2.0,Living,0,164.933333
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,1,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,0,Positive,1,ER+/HER2- High Prolif,40.0,2.0,Died of Disease,1,41.366667


In [56]:
df[["3-gene_classifier_subtype"]].value_counts()

3-gene_classifier_subtype
ER+/HER2- Low Prolif         619
ER+/HER2- High Prolif        603
ER-/HER2-                    290
unknown                      204
HER2+                        188
Name: count, dtype: int64

In [57]:
# group rare cancer types into "Other" category
# This will make one hot encoding and modeling more robust by reducing the number of categories with very few samples
rare_threshold = 50

value_counts = df["cancer_type_detailed"].value_counts()

rare_classes = value_counts[value_counts < rare_threshold].index

df["cancer_type_detailed_clean"] = df["cancer_type_detailed"].replace(
    rare_classes, "Other"
)

# do same for histologic subtype
value_counts = df["tumor_other_histologic_subtype"].value_counts()

rare_classes = value_counts[value_counts < rare_threshold].index

df["tumor_other_histologic_subtype_clean"] = df["tumor_other_histologic_subtype"].replace(
    rare_classes, "Other"
)

In [58]:
df[["cancer_type_detailed","cancer_type_detailed_clean"]].value_counts()

cancer_type_detailed                       cancer_type_detailed_clean               
Breast Invasive Ductal Carcinoma           Breast Invasive Ductal Carcinoma             1500
Breast Mixed Ductal and Lobular Carcinoma  Breast Mixed Ductal and Lobular Carcinoma     207
Breast Invasive Lobular Carcinoma          Breast Invasive Lobular Carcinoma             142
Breast Invasive Mixed Mucinous Carcinoma   Other                                          22
Breast                                     Other                                          17
Metaplastic Breast Cancer                  Other                                           1
Name: count, dtype: int64

In [59]:
df[["tumor_other_histologic_subtype","tumor_other_histologic_subtype_clean"]].value_counts()

tumor_other_histologic_subtype  tumor_other_histologic_subtype_clean
Ductal/NST                      Ductal/NST                              1454
Mixed                           Mixed                                    207
Lobular                         Lobular                                  142
Medullary                       Other                                     25
Mucinous                        Other                                     22
Tubular/ cribriform             Other                                     21
Other                           Other                                     17
Metaplastic                     Other                                      1
Name: count, dtype: int64

In [60]:
df

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,er_status,neoplasm_histologic_grade,her2_status_measured_by_snp6,her2_status,tumor_other_histologic_subtype,hormone_therapy,inferred_menopausal_state,integrative_cluster,primary_tumor_laterality,lymph_nodes_examined_positive,mutation_count,nottingham_prognostic_index,oncotree_code,overall_survival_months,overall_survival,pr_status,radio_therapy,3-gene_classifier_subtype,tumor_size,tumor_stage,death_from_cancer,event,time,cancer_type_detailed_clean,tumor_other_histologic_subtype_clean
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,1,Negative,1,ER-/HER2-,22.0,2.0,Living,0,140.500000,Breast Invasive Ductal Carcinoma,Ductal/NST
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,1,Positive,1,ER+/HER2- High Prolif,10.0,1.0,Living,0,84.633333,Breast Invasive Ductal Carcinoma,Ductal/NST
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,1,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,0,Positive,0,unknown,15.0,2.0,Died of Disease,1,163.700000,Breast Invasive Ductal Carcinoma,Ductal/NST
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,1,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,1,Positive,1,unknown,25.0,2.0,Living,0,164.933333,Breast Mixed Ductal and Lobular Carcinoma,Mixed
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,1,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,0,Positive,1,ER+/HER2- High Prolif,40.0,2.0,Died of Disease,1,41.366667,Breast Mixed Ductal and Lobular Carcinoma,Mixed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,7295,43.10,BREAST CONSERVING,Breast Cancer,Breast Invasive Lobular Carcinoma,High,0,LumA,4.0,Positve,Positive,3.0,NEUTRAL,Negative,Lobular,1,Pre,3,Right,1.0,4.0,5.050,ILC,196.866667,1,Positive,1,ER+/HER2- Low Prolif,25.0,unknown,Living,0,196.866667,Breast Invasive Lobular Carcinoma,Lobular
1900,7296,42.88,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,Positive,3.0,GAIN,Positive,Ductal/NST,0,Pre,5,NaN,1.0,6.0,5.040,IDC,44.733333,0,Negative,1,unknown,20.0,unknown,Died of Disease,1,44.733333,Breast Invasive Ductal Carcinoma,Ductal/NST
1901,7297,62.90,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Post,1,Left,45.0,4.0,6.050,IDC,175.966667,0,Positive,1,unknown,25.0,unknown,Died of Disease,1,175.966667,Breast Invasive Ductal Carcinoma,Ductal/NST
1902,7298,61.16,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,Moderate,0,LumB,4.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,1,Post,1,NaN,12.0,15.0,5.050,IDC,86.233333,0,Positive,0,ER+/HER2- High Prolif,25.0,unknown,Died of Other Causes,0,86.233333,Breast Invasive Ductal Carcinoma,Ductal/NST


In [61]:
# Drop identifiers, outcome columns, and pre-modified columns from clinical features
drop_cols = [
    # original columns to drop
    "patient_id",
    "cancer_type",
    "cohort",
    "cancer_type_detailed",
    "er_status_measured_by_ihc",
    "her2_status_measured_by_snp6",
    "tumor_other_histologic_subtype",
    "integrative_cluster",
    "oncotree_code",
    "radio_therapy",
    "chemotherapy",
    "type_of_breast_surgery",
    "hormone_therapy", 
    "overall_survival",
    # updated drop columns 
    "nottingham_prognostic_index",  # composite of existing features
    "3-gene_classifier_subtype",    # derived from ER/PR/HER2
    "tumor_other_histologic_subtype_clean",  # redundant with cancer_type_detailed
    "primary_tumor_laterality",
    time_col,
    event_col
]

# X will now contain only clinical features that will go into model training
X = df.drop(columns=drop_cols + ["event", "time"])
y_time = df["time"].values
y_event = df["event"].values

#### Identify feature types

In [62]:
# different data types in clinical data
X.dtypes.value_counts()

object     8
float64    5
Name: count, dtype: int64

In [63]:
# identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# fill missing categorical data with "Unknown"
X[categorical_cols] = X[categorical_cols].fillna("Unknown")
numeric_cols
categorical_cols

['cellularity',
 'pam50_+_claudin-low_subtype',
 'er_status',
 'her2_status',
 'inferred_menopausal_state',
 'pr_status',
 'tumor_stage',
 'cancer_type_detailed_clean']

### Train/Validation Split

In [64]:
from sklearn.model_selection import train_test_split

# First split off test set (hold this out entirely until the end)
X_train, X_test, time_train, time_test, event_train, event_test = train_test_split(
    X, y_time, y_event,
    test_size=0.20,
    stratify=y_event,
    random_state=42
)

#### Process numeric features for training and validation set separately
Scaling/imputation depends on the data that we see, so we must normalize the data separately for training and validation to prevent data leakage

In [65]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_num = num_imputer.fit_transform(X_train[numeric_cols])
X_train_num = scaler.fit_transform(X_train_num)

X_test_num = num_imputer.transform(X_test[numeric_cols])
X_test_num = scaler.transform(X_test_num)

#### Process categorical features separately 
One-hot encode separately

In [66]:
X_train_cat = pd.get_dummies(
    X_train[categorical_cols],
    drop_first=True
).astype(float)

X_test_cat = pd.get_dummies(
    X_test[categorical_cols],
    drop_first=True
).astype(float)

X_test_cat = X_test_cat.reindex(
    columns=X_train_cat.columns,
    fill_value=0
)

In [67]:
# Drop identifiers, outcome columns, and pre-modified columns from clinical features
drop_cols = [
    # original columns to drop
    "cellularity_Unknown", 
    "tumor_stage_unknown",
    "cancer_type_detailed_clean_Unknown"
]

# X will now contain only clinical features that will go into model training
X_train_cat = X_train_cat.drop(columns=drop_cols)
X_test_cat = X_test_cat.drop(columns=drop_cols)

In [68]:
X_train_cat

,cellularity_Low,cellularity_Moderate,pam50_+_claudin-low_subtype_Her2,pam50_+_claudin-low_subtype_LumA,pam50_+_claudin-low_subtype_LumB,pam50_+_claudin-low_subtype_NC,pam50_+_claudin-low_subtype_Normal,pam50_+_claudin-low_subtype_claudin-low,er_status_Positive,her2_status_Positive,inferred_menopausal_state_Pre,pr_status_Positive,tumor_stage_1.0,tumor_stage_2.0,tumor_stage_3.0,tumor_stage_4.0,cancer_type_detailed_clean_Breast Invasive Lobular Carcinoma,cancer_type_detailed_clean_Breast Mixed Ductal and Lobular Carcinoma,cancer_type_detailed_clean_Other
505,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1080,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
956,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
607,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1894,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1278,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
450,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
35,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1787,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
X_train_num

array([[-0.47989657, -2.20848878, -0.25213263,  0.07501381, -1.00838441],
       [ 0.33469459,  0.87166544, -0.49228147, -0.16601157, -0.54725153],
       [-0.65714914,  0.87166544, -0.25213263,  0.07501381,  1.29727999],
       ...,
       [-1.36153544,  0.87166544, -0.49228147, -0.88908772, -0.54725153],
       [-0.29956135,  0.87166544,  3.35009994,  0.31603919, -0.02024253],
       [-0.95154254, -2.20848878, -0.49228147,  0.07501381, -0.74487991]],
      shape=(1523, 5))

#### Combine features

In [70]:
import numpy as np

X_train_final = np.hstack([
    X_train_num.astype(np.float32),
    X_train_cat.values.astype(np.float32)
])

X_test_final = np.hstack([
    X_test_num.astype(np.float32),
    X_test_cat.values.astype(np.float32)
])

#### Convert to PyTorch Tensors

In [71]:
import torch

X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
time_train_tensor = torch.tensor(time_train, dtype=torch.float32)
event_train_tensor = torch.tensor(event_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32)
time_test_tensor = torch.tensor(time_test, dtype=torch.float32)
event_test_tensor = torch.tensor(event_test, dtype=torch.float32)

## Create Survival-Aware Autoencoder

#### To optimize final results, we will try expanding our autoencoder in 2 ways: 
##### 1. Survival-aware autoencoder pretraining
Instead of pure reconstruction loss, pretrain the AE with combined loss of reconstruction (stander AE loss) + smaller Cox survival loss. This pushes the autoencoder to encode survival-relevant variation instead of just expression variance. 
##### 2. Larger latent dimension 
Current latent dims = 16, we will increase this to 32 or 64 so the autoencoder has more dimensions to encode variation

#### Model training

In [73]:
from lifelines.utils import concordance_index
import torch

def compute_cindex(model, X_tensor, time_array, event_array):
    """
    Compute the concordance index for a PyTorch model.
    
    Parameters:
        model : torch.nn.Module
            Trained survival model that outputs risk scores
        X_tensor : torch.Tensor
            Input features
        time_array : np.array
            Survival times
        event_array : np.array
            Event indicator (1=event, 0=censored)
            
    Returns:
        c_index : float
            Concordance index
    """
    model.eval()
    with torch.no_grad():
        risk_scores = model(X_tensor).numpy()
    
    return concordance_index(time_array, -risk_scores, event_array)

In [74]:
import numpy as np

# helper function to convert event/time arrays to structured array format for sksurv compatibility
def make_structured_array(event, time):
    """Convert event/time arrays to sksurv structured array format"""
    return np.array(
        [(bool(e), t) for e, t in zip(event, time)],
        dtype=[('event', bool), ('time', float)]
    )

In [75]:
def compute_ibs(train_risk, val_risk, surv_train, surv_val, time_val):
    """
    Compute the Integrated Brier Score for a survival model.

    Parameters:
        train_risk  : np.array — risk scores for training set
        val_risk    : np.array — risk scores for validation/test set
        surv_train  : structured array — training survival data (event, time)
        surv_val    : structured array — validation survival data (event, time)
        time_val    : np.array — survival times for validation/test set

    Returns:
        ibs : float — Integrated Brier Score
    """
    from sksurv.metrics import integrated_brier_score
    from sksurv.linear_model import CoxPHSurvivalAnalysis

    # Fit Cox reference on training risk scores to get baseline hazard
    cox_ref = CoxPHSurvivalAnalysis()
    cox_ref.fit(train_risk.reshape(-1, 1), surv_train)

    # Time grid within observed range
    times_ibs = np.linspace(
        time_val.min() + 1,
        np.percentile(time_val, 90),
        100
    )

    # Get survival probabilities for each patient at each time point
    surv_fns = cox_ref.predict_survival_function(val_risk.reshape(-1, 1))
    surv_probs = np.vstack([fn(times_ibs) for fn in surv_fns])

    ibs = integrated_brier_score(surv_train, surv_val, surv_probs, times_ibs)
    return ibs

In [ ]:
# ── Survival-aware Autoencoder ────────────────────────────────────

class SurvivalAwareAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

        # Small survival head attached to latent space
        # Guides the encoder to preserve survival-relevant variation
        self.survival_head = nn.Linear(latent_dim, 1)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        risk = self.survival_head(z).squeeze(-1)
        return x_recon, z, risk

# Define loss function first
recon_fn = nn.MSELoss()

# First measure the natural scale of each loss before training
ae_test = SurvivalAwareAutoencoder(input_dim=expr_dim, latent_dim=64)
ae_test.eval()
with torch.no_grad():
    recon_test, _, risk_test = ae_test(expr_train_tensor)
    r_loss = recon_fn(recon_test, expr_train_tensor).item()
    s_loss = cox_loss(risk_test, time_train_tensor, event_train_tensor).item()

print(f"Initial recon loss: {r_loss:.4f}")
print(f"Initial surv loss:  {s_loss:.4f}")
print(f"Natural ratio:      {s_loss/r_loss:.2f}x")

# Set weight to compensate for the scale difference
# We want surv to contribute ~5% of total loss
surv_weight = 0.05 * (r_loss / s_loss)
print(f"Corrected surv_weight: {surv_weight:.4f}")


# ── Pretraining with combined loss ───────────────────────────────
set_seed(42)

print("Pretraining survival-aware autoencoder (latent_dim=32)...")

expr_dim  = expr_train_tensor.shape[1]
ae_surv   = SurvivalAwareAutoencoder(input_dim=expr_dim, latent_dim=64)
ae_opt    = torch.optim.Adam(ae_surv.parameters(), lr=1e-3, weight_decay=1e-4)
recon_fn  = nn.MSELoss()

# Survival weight — start small so reconstruction drives learning
# then the latent space gradually becomes survival-aware

#surv_weight      = 0.1   # set above
best_val_recon   = float('inf')
best_ae_weights  = None
ae_patience      = 30
ae_no_improve    = 0
best_combined = 0

for epoch in range(500):
    ae_surv.train()
    ae_opt.zero_grad()

    recon, z, risk = ae_surv(expr_train_tensor)

    recon_loss = recon_fn(recon, expr_train_tensor)
    surv_loss  = cox_loss(risk, time_train_tensor, event_train_tensor)

    # Warmup: no survival loss for first 100 epochs
    # then linearly ramp up to corrected weight over next 100 epochs
    if epoch < 100:
        effective_weight = 0.0
    elif epoch < 200:
        effective_weight = surv_weight * (epoch - 100) / 100
    else:
        effective_weight = surv_weight

    total_loss = recon_loss + effective_weight * surv_loss
    total_loss.backward()
    ae_opt.step()

    if epoch % 10 == 0:
        ae_surv.eval()
        with torch.no_grad():
            val_recon, _, val_risk = ae_surv(expr_val_tensor)
            val_recon_loss = recon_fn(val_recon, expr_val_tensor).item()
            val_ci = concordance_index(
                time_val, -val_risk.numpy(), event_val
            )

        # Stop on val C-index rather than reconstruction
        # Target: val C-index between 0.58-0.65 with low recon loss
        combined_score = val_ci - val_recon_loss  # balance both

        if combined_score > best_combined:
            best_combined   = combined_score
            best_ae_weights = {k: v.clone() for k, v in ae_surv.state_dict().items()}
            ae_no_improve   = 0
        else:
            ae_no_improve  += 1

        if ae_no_improve >= ae_patience:
            print(f"  Early stopping at epoch {epoch}")
            break

        if epoch % 50 == 0:
            print(f"  Epoch {epoch:>4} | "
                  f"Recon: {recon_loss.item():.4f} | "
                  f"Surv:  {surv_loss.item():.4f} | "
                  f"Val recon: {val_recon_loss:.4f} | "
                  f"Val C-index: {val_ci:.4f}")

ae_surv.load_state_dict(best_ae_weights)
print(f"\nBest val reconstruction loss: {best_val_recon:.4f}")


# ── Verify latent space has survival signal ───────────────────────
ae_surv.eval()
with torch.no_grad():
    _, _, train_risk_ae = ae_surv(expr_train_tensor)
    _, _, val_risk_ae   = ae_surv(expr_val_tensor)

latent_train_ci = concordance_index(
    time_train, -train_risk_ae.numpy(), event_train
)
latent_val_ci = concordance_index(
    time_val, -val_risk_ae.numpy(), event_val
)
print(f"Latent space C-index (train): {latent_train_ci:.4f}")
print(f"Latent space C-index (val):   {latent_val_ci:.4f}")
# A val C-index > 0.55 confirms the latent space captured survival signal


# ── Updated survival model using new AE ──────────────────────────

class ClinicalExprSurvAESurvNet(nn.Module):
    """
    Same architecture as ClinicalExprAESurvNet but accepts
    the larger latent_dim=32 encoder output.
    """
    def __init__(self, clin_dim, pretrained_encoder, latent_dim=32):
        super().__init__()

        self.clin_enc = nn.Sequential(
            nn.Linear(clin_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.expr_enc = pretrained_encoder

        # 16 (clin) + latent_dim (expr) → fusion
        self.head = nn.Sequential(
            nn.Linear(16 + latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_clin, x_expr):
        z_clin = self.clin_enc(x_clin)
        z_expr = self.expr_enc(x_expr)
        z = torch.cat([z_clin, z_expr], dim=1)
        return self.head(z).squeeze(-1)


# ── Phase 1: frozen encoder ───────────────────────────────────────
model_surv_ae = ClinicalExprSurvAESurvNet(
    clin_dim=X_train_tensor.shape[1],
    pretrained_encoder=ae_surv.encoder,
    latent_dim=64
)

for param in model_surv_ae.expr_enc.parameters():
    param.requires_grad = False

optimizer_p1 = torch.optim.Adam([
    {"params": model_surv_ae.clin_enc.parameters(), "lr": 1e-3},
    {"params": model_surv_ae.head.parameters(),     "lr": 1e-3}
], weight_decay=1e-3)

best_val_cindex   = 0
patience          = 50
epochs_no_improve = 0
best_weights      = None

print("\nPhase 1: Training with frozen encoder...")

for epoch in range(500):
    model_surv_ae.train()
    optimizer_p1.zero_grad()
    risk = model_surv_ae(X_train_tensor, expr_train_tensor)
    loss = cox_loss(risk, time_train_tensor, event_train_tensor)
    loss.backward()
    optimizer_p1.step()

    if epoch % 10 == 0:
        val_ci = compute_cindex(
            model_surv_ae, time_val, event_val,
            X_val_tensor, expr_val_tensor
        )
        if val_ci > best_val_cindex:
            best_val_cindex   = val_ci
            best_weights      = {
                k: v.clone() for k, v in model_surv_ae.state_dict().items()
            }
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break

        if epoch % 100 == 0:
            print(f"  Epoch {epoch} | Val C-index: {val_ci:.4f}")

model_surv_ae.load_state_dict(best_weights)
print(f"Phase 1 best val C-index: {best_val_cindex:.4f}")


# ── Phase 2: unfreeze and fine-tune ──────────────────────────────
for param in model_surv_ae.expr_enc.parameters():
    param.requires_grad = True

optimizer_p2 = torch.optim.Adam([
    {"params": model_surv_ae.clin_enc.parameters(), "lr": 1e-4},
    {"params": model_surv_ae.expr_enc.parameters(), "lr": 1e-5},
    {"params": model_surv_ae.head.parameters(),     "lr": 1e-4}
], weight_decay=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode='max', factor=0.5, patience=20, min_lr=1e-7
)

best_val_cindex_ft = best_val_cindex
patience           = 100
epochs_no_improve  = 0

print("\nPhase 2: Fine-tuning all layers...")

for epoch in range(2000):
    model_surv_ae.train()
    optimizer_p2.zero_grad()
    risk = model_surv_ae(X_train_tensor, expr_train_tensor)
    loss = cox_loss(risk, time_train_tensor, event_train_tensor)
    loss.backward()
    optimizer_p2.step()

    if epoch % 10 == 0:
        val_ci = compute_cindex(
            model_surv_ae, time_val, event_val,
            X_val_tensor, expr_val_tensor
        )
        scheduler.step(val_ci)

        if val_ci > best_val_cindex_ft:
            best_val_cindex_ft = val_ci
            best_weights       = {
                k: v.clone() for k, v in model_surv_ae.state_dict().items()
            }
            epochs_no_improve  = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break

        if epoch % 100 == 0:
            lr = optimizer_p2.param_groups[0]['lr']
            print(f"  Epoch {epoch} | "
                  f"Val C-index: {val_ci:.4f} | "
                  f"LR: {lr:.2e}")

model_surv_ae.load_state_dict(best_weights)
print(f"Phase 2 best val C-index: {best_val_cindex_ft:.4f}")


# ── Evaluate and compare ──────────────────────────────────────────
model_surv_ae.eval()
with torch.no_grad():
    train_risk_surv_ae = model_surv_ae(
        X_train_tensor, expr_train_tensor
    ).numpy()

val_results_surv_ae = evaluate_model(
    model_surv_ae, time_val, event_val, surv_train,
    [X_val_tensor, expr_val_tensor],
    train_risk=train_risk_surv_ae,
    label="Validation (Surv-Aware AE, latent=32)"
)

test_results_surv_ae = evaluate_model(
    model_surv_ae, time_test, event_test, surv_train,
    [X_test_tensor, expr_test_tensor],
    train_risk=train_risk_surv_ae,
    label="Test (Surv-Aware AE, latent=32)"
)

# Compare against previous best
print(f"\n{'='*65}")
print(f"  Model Comparison (Test Set)")
print(f"{'='*65}")
print(f"  {'Metric':<22} {'Clin+AE (16)':>14} {'Clin+SurvAE (32)':>16}")
print(f"  {'-'*55}")
for metric, label, better in [
    ("c_index",  "C-index",  "↑"),
    ("mean_auc", "Mean AUC", "↑"),
    ("ibs",      "IBS",      "↓"),
]:
    prev = test_results_ae[metric]
    curr = test_results_surv_ae[metric]
    delta = curr - prev if better == "↑" else prev - curr
    sign  = "+" if delta > 0 else ""
    print(f"  {label:<22} {prev:>14.4f} {curr:>16.4f}  "
          f"({sign}{delta:.4f})")

Initial recon loss: 1.0016
Initial surv loss:  6.6678
Natural ratio:      6.66x
Corrected surv_weight: 0.0075
Pretraining survival-aware autoencoder (latent_dim=32)...
  Epoch    0 | Recon: 1.0016 | Surv:  6.6665 | Val recon: 0.9800 | Val C-index: 0.5741
  Epoch   50 | Recon: 0.7464 | Surv:  6.6961 | Val recon: 0.7269 | Val C-index: 0.4677
  Epoch  100 | Recon: 0.6710 | Surv:  6.6643 | Val recon: 0.6646 | Val C-index: 0.5487
  Epoch  150 | Recon: 0.6289 | Surv:  6.5674 | Val recon: 0.6362 | Val C-index: 0.6881
  Epoch  200 | Recon: 0.6009 | Surv:  6.5070 | Val recon: 0.6253 | Val C-index: 0.7003
  Epoch  250 | Recon: 0.5806 | Surv:  6.4617 | Val recon: 0.6218 | Val C-index: 0.7007
  Epoch  300 | Recon: 0.5659 | Surv:  6.4171 | Val recon: 0.6224 | Val C-index: 0.7030
  Epoch  350 | Recon: 0.5534 | Surv:  6.3347 | Val recon: 0.6228 | Val C-index: 0.6945
  Epoch  400 | Recon: 0.5433 | Surv:  6.2586 | Val recon: 0.6258 | Val C-index: 0.6976
  Epoch  450 | Recon: 0.5323 | Surv:  6.1305 | Va